# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [2]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [6]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [14]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [15]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.7769742620704805)),
 (1, np.float64(2.7769742620704805)),
 (2, np.float64(2.7769742620704805)),
 (3, np.float64(2.7769742620704805)),
 (4, np.float64(2.7769742620704805))]

## Inverted index

In [16]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [17]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('what', 10)]),
 (1, [('a', 2), ('is', 18), ('it', 18)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.1403743362959765)),
   (None, np.float64(0.16770373874831113)),
   (None, np.float64(0.23718494158752212)),
   (None, np.float64(0.24068347808298363)),
   (None, np.float64(0.24769112184497133)),
   (None, np.float64(0.2827766491012139)),
   (None, np.float64(0.31831172983340483)),
   (None, np.float64(0.3539069373003818)),
   (None, np.float64(0.3573301310050485)),
   (None, np.float64(0.36734795399907694)),
   (None, np.float64(0.3923149390235905)),
   (None, np.float64(0.41683003158622345)),
   (None, np.float64(0.4536396412497641))]),
 (1,
  [(None, np.float64(0.5415085267091696)),
   (None, np.float64(0.5415204373144457)),
   (None, np.float64(0.5662444698078719)),
   (None, np.float64(0.5990743145589814)),
   (None, np.float64(0.623589292797202)),
   (None, np.float64(0.625650489569925)),
   (None, np.float64(0.7391544146022497)),
   (None, np.float64(0.7714535742424877)),
   (None, np.float64(0.7726561903316972)),
   (None, np.float64(0.81443316665672

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [21]:
input_list = [4, 10, 3, 25, 1, 17]

def RECORDREADER():
    for i, val in enumerate(input_list):
        yield (i, val)

def MAP(k1, v1):
    yield ("max", v1)

def REDUCE(key, values):
    yield (key, max(values))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('max', 25)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [22]:
input_list = [10, 20, 30, 40]

def RECORDREADER():
    for i, val in enumerate(input_list):
        yield (i, val)

def MAP(k1, v1):
    yield ("avg", v1)

def REDUCE(key, values):
    sum_val = 0
    count = 0
    for v in values:
        sum_val += v
        count += 1
    yield (key, sum_val / count if count > 0 else 0)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print(output)

[('avg', 25.0)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [23]:
def groupbykey_sort(iterable):
    sorted_iterable = sorted(iterable, key=lambda x: x[0])

    t = {}
    for k2, v2 in sorted_iterable:
        if k2 not in t:
            t[k2] = []
        t[k2].append(v2)
    return t.items()

test_data = [(25, 'user1'), (33, 'user3'), (25, 'user2')]
print(list(groupbykey_sort(test_data)))

[(25, ['user1', 'user2']), (33, ['user3'])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [24]:
data = ['apple', 'banana', 'apple', 'orange', 'banana']

def RECORDREADER():
    for i, val in enumerate(data):
        yield (i, val)

def MAP(k1, v1):
    yield (v1, None)

def REDUCE(key, values):
    yield (key, None)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
unique_elements = [k for k, v in output]
print(unique_elements)

['apple', 'banana', 'orange']


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [25]:
input_data = [1, 2, 3, 4, 5]

def RECORDREADER():
    for val in input_data:
        yield (val, val)

def MAP(k1, v1):
    if v1 > 2:
        yield (v1, v1)

def REDUCE(key, values):
    yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Selection:", output)

Selection: [(3, 3), (4, 4), (5, 5)]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [26]:
input_tuples = [(1, 'x', 10), (2, 'y', 20), (1, 'z', 10)]

def RECORDREADER():
    for t in input_tuples:
        yield (t, t)

def MAP(k1, v1):
    t_prime = (v1[0], v1[2])
    yield (t_prime, t_prime)

def REDUCE(key, values):
    yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Projection:", output)

Projection: [((1, 10), (1, 10)), ((2, 20), (2, 20))]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [27]:
R = [1, 2, 3]
S = [3, 4, 5]

def RECORDREADER():
    for val in R + S:
        yield (val, val)

def MAP(k1, v1):
    yield (v1, v1)

def REDUCE(key, values):
    yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Union:", output)

Union: [(1, 1), (2, 2), (3, 3), (4, 4), (5, 5)]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [28]:
R = [1, 2, 3]
S = [3, 4, 5]

def RECORDREADER():
    for val in R + S:
        yield (val, val)

def MAP(k1, v1):
    yield (v1, v1)

def REDUCE(key, values):
    if len(values) == 2:
        yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Intersection:", output)

Intersection: [(3, 3)]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [29]:
R = [1, 2, 3]
S = [3, 4, 5]

def RECORDREADER():
    for val in R: yield (val, 'R')
    for val in S: yield (val, 'S')

def MAP(k1, v1):
    yield (k1, v1)

def REDUCE(key, values):
    if values == ['R']:
        yield (key, key)

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Difference (R - S):", output)

Difference (R - S): [(1, 1), (2, 2)]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [30]:
R = [('a', 1), ('b', 2)]
S = [(1, 'x'), (1, 'y'), (2, 'z')]

def RECORDREADER():
    for a, b in R: yield (None, ('R', a, b))
    for b, c in S: yield (None, ('S', b, c))

def MAP(k1, v1):
    rel_name = v1[0]
    if rel_name == 'R':
        yield (v1[2], ('R', v1[1]))
    else:
        yield (v1[1], ('S', v1[2]))

def REDUCE(key_b, values):
    r_a_vals = [val[1] for val in values if val[0] == 'R']
    s_c_vals = [val[1] for val in values if val[0] == 'S']

    for a in r_a_vals:
        for c in s_c_vals:
            yield (key_b, (a, key_b, c))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Natural Join:", output)

Natural Join: [(1, ('a', 1, 'x')), (1, ('a', 1, 'y')), (2, ('b', 2, 'z'))]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [31]:
input_data = [('A', 1, 10), ('A', 2, 20), ('B', 1, 5)]

def RECORDREADER():
    for t in input_data: yield (None, t)

def MAP(k1, v1):
    a, b, c = v1
    yield (a, c)

def REDUCE(key, values):
    yield (key, sum(values))

output = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Grouping and Aggregation (SUM):", output)

Grouping and Aggregation (SUM): [('A', 30), ('B', 5)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [32]:
M = [(0, 0, 1), (0, 1, 2), (1, 0, 3), (1, 1, 4)]
V = [(0, 5), (1, 6)]

def RECORDREADER():
    for i, j, v in M: yield (None, ('M', i, j, v))
    for j, w in V: yield (None, ('V', j, w))

def MAP(k1, v1):
    if v1[0] == 'M':
        yield (v1[2], ('M', v1[1], v1[3]))
    else:
        yield (v1[1], ('V', v1[2]))

def REDUCE(key_j, values):
    v_val = 0
    m_vals = []
    for val in values:
        if val[0] == 'V':
            v_val = val[1]
        else:
            m_vals.append((val[1], val[2]))

    for i, m_ij in m_vals:
        yield (i, m_ij * v_val)

output_stage_1 = list(MapReduce(RECORDREADER, MAP, REDUCE))
print("Matrix-Vector Stage 1 (products):", output_stage_1)

Matrix-Vector Stage 1 (products): [(0, 5), (1, 15), (0, 12), (1, 24)]


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [33]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [34]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
  (j, k) = k1
  w = v1
  for i in range(I):
      yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
  (i, k) = key
  yield (key, sum(values))

Проверьте своё решение

In [35]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [36]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [37]:
import numpy as np
I, J, K = 2, 3, 4
M_mat = np.random.rand(I, J)
N_mat = np.random.rand(J, K)

def RECORDREADER_JOIN():
    for i in range(I):
        for j in range(J):
            yield (None, ('M', i, j, M_mat[i, j]))
    for j in range(J):
        for k in range(K):
            yield (None, ('N', j, k, N_mat[j, k]))

def MAP_JOIN(k1, v1):
    rel = v1[0]
    if rel == 'M':
        yield (v1[2], ('M', v1[1], v1[3]))
    else:
        yield (v1[1], ('N', v1[2], v1[3]))

def REDUCE_JOIN(key_j, values):
    m_vals = [v for v in values if v[0] == 'M']
    n_vals = [v for v in values if v[0] == 'N']
    for m in m_vals:
        for n in n_vals:
            yield ((m[1], n[1]), m[2] * n[2])

stage_1_output = list(MapReduce(RECORDREADER_JOIN, MAP_JOIN, REDUCE_JOIN))

def RECORDREADER_AGG():
    for k, v in stage_1_output:
        yield (k, v)

def MAP_AGG(k1, v1):
    yield (k1, v1)

def REDUCE_AGG(key_ik, values):
    yield (key_ik, sum(values))

final_output = list(MapReduce(RECORDREADER_AGG, MAP_AGG, REDUCE_AGG))
print("Two-stage Matrix Mult Output length:", len(final_output))

Two-stage Matrix Mult Output length: 8


Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [38]:
import numpy as np
I, J, K = 2, 3, 4
M_mat = np.random.rand(I, J)
N_mat = np.random.rand(J, K)

reducers = 2

def INPUTFORMAT():
    def RECORDREADER_M():
        for i in range(I):
            for j in range(J):
                yield (None, ('M', i, j, M_mat[i, j]))

    def RECORDREADER_N():
        for j in range(J):
            for k in range(K):
                yield (None, ('N', j, k, N_mat[j, k]))

    yield RECORDREADER_M()
    yield RECORDREADER_N()

def MAP(k1, v1):
    rel = v1[0]
    if rel == 'M':
        _, i, j, val = v1
        for k in range(K):
            yield ((i, k), ('M', j, val))
    else:
        _, j, k, val = v1
        for i in range(I):
            yield ((i, k), ('N', j, val))

def REDUCE(key, values):
    m_dict = {}
    n_dict = {}

    for val in values:
        if val[0] == 'M':
            m_dict[val[1]] = val[2]
        else:
            n_dict[val[1]] = val[2]

    cell_sum = 0
    for j in range(J):
        if j in m_dict and j in n_dict:
            cell_sum += m_dict[j] * n_dict[j]

    yield (key, cell_sum)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)

flat_output = list(flatten([partition for (partition_id, partition) in partitioned_output]))
print("Distributed Output:", flat_output)

48 key-value pairs were sent over a network.
Distributed Output: [((0, 1), np.float64(1.021035094290658)), ((0, 2), np.float64(0.5816837372139181)), ((1, 0), np.float64(0.6573296285156387)), ((1, 1), np.float64(0.5858653406859521)), ((1, 3), np.float64(0.654789852962613)), ((0, 0), np.float64(1.2989791812771665)), ((0, 3), np.float64(1.1537698059588475)), ((1, 2), np.float64(0.3805408892877038))]


Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [39]:
import numpy as np
import random

I, J, K = 2, 3, 4
M_mat = np.random.rand(I, J)
N_mat = np.random.rand(J, K)

reducers = 3
maps = 4

all_elements = []
for i in range(I):
    for j in range(J):
        all_elements.append(('M', i, j, M_mat[i, j]))
for j in range(J):
    for k in range(K):
        all_elements.append(('N', j, k, N_mat[j, k]))

random.shuffle(all_elements)

def INPUTFORMAT():
    global maps
    def RECORDREADER(split):
        for elem in split:
            yield (None, elem)

    split_size = int(np.ceil(len(all_elements) / maps))
    for i in range(0, len(all_elements), split_size):
        yield RECORDREADER(all_elements[i : i + split_size])

partitioned_output_2 = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)

reduce_output = list(flatten([partition for (partition_id, partition) in partitioned_output_2]))

result_mat = np.zeros((I, K))
for (i, k), val in reduce_output:
    result_mat[i, k] = val

reference_solution = np.matmul(M_mat, N_mat)
is_correct = np.allclose(reference_solution, result_mat)

print("\nРезультат совпадает с np.matmul:", is_correct)

48 key-value pairs were sent over a network.

Результат совпадает с np.matmul: True
